# Automatic Parameter Selection

Demonstrates the G2-based automatic parameter selection for RM3's two thresholds:

1. **RM1 size threshold** — calibrated on watershed labels
2. **RM2 cost threshold** — calibrated on *post-RM1* labels (not raw watershed labels!)

G2 = normalised(Moran's I) + normalised(intra-variance). Lower G2 = better segmentation.

Key detail: the sweep uses the **filtered** image for merging costs but the **original** image for G2 evaluation (matching the MATLAB implementation).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from watershed_seg.filters import epsf
from watershed_seg.watershed_input import h_image
from watershed_seg.watershed import vincent_soille
from watershed_seg.merging import rm1, rm2, rm3
from watershed_seg.evaluation import morans_i, intra_variance, goodness2
from watershed_seg.auto_params import auto_select_rm1_threshold, auto_select_rm2_threshold

# Build synthetic image with noise to make filtering meaningful
rng = np.random.default_rng(1)
img = np.zeros((64, 64, 3), dtype=np.float64)
img[:32, :32] = rng.uniform(180, 240, (32, 32, 3))  # wider range = more noise
img[:32, 32:] = rng.uniform(30,  90,  (32, 32, 3))
img[32:, :32] = rng.uniform(80,  140, (32, 32, 3))
img[32:, 32:] = rng.uniform(130, 190, (32, 32, 3))
# Add salt-and-pepper noise
noise_mask = rng.random(img.shape[:2]) < 0.05
img[noise_mask] = rng.choice([0, 255], size=(noise_mask.sum(), 3))

# Pre-filter and build watershed
filtered = epsf(img, w=5)
gradient = h_image(filtered, w=7)
ws_labels = vincent_soille(gradient)
print(f'Image noise std: {img.std():.1f} → filtered std: {filtered.std():.1f}')
print(f'Watershed: {ws_labels.max()} initial segments')

In [ ]:
# --- Step 1: Auto-select RM1 threshold ---
# Sweep uses filtered image for merging, original image for G2 evaluation
search_rm1 = list(range(1, 51))
all_mi, all_var, seg_counts = [], [], []

for t in search_rm1:
    labels = rm1(ws_labels, filtered, size_threshold=t)
    all_mi.append(morans_i(labels, img))
    all_var.append(intra_variance(labels, img))
    seg_counts.append(int(labels.max()))

mi_arr = np.array(all_mi).T
var_arr = np.array(all_var).T
g2_rm1 = goodness2(mi_arr, var_arr)

best_rm1_idx = int(np.argmin(g2_rm1))
best_rm1 = search_rm1[best_rm1_idx]
print(f'Auto-selected RM1 threshold: {best_rm1} (G2={g2_rm1[best_rm1_idx]:.4f})')

# Apply RM1 to get post-RM1 labels (needed for RM2 calibration)
rm1_labels = rm1(ws_labels, filtered, size_threshold=best_rm1)
print(f'After RM1: {rm1_labels.max()} segments (from {ws_labels.max()})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(search_rm1, g2_rm1, 'b-o', markersize=3)
axes[0].axvline(x=best_rm1, color='r', linewidth=2, label=f'Auto-selected={best_rm1}')
axes[0].set_xlabel('RM1 size threshold')
axes[0].set_ylabel('G2 (lower = better)')
axes[0].set_title('G2 curve — RM1 threshold sweep')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(search_rm1, seg_counts, 'g-s', markersize=3)
axes[1].axvline(x=best_rm1, color='r', linewidth=2, label=f'Auto-selected={best_rm1}')
axes[1].set_xlabel('RM1 size threshold')
axes[1].set_ylabel('Number of segments')
axes[1].set_title('Segment count vs. RM1 threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 2: Auto-select RM2 threshold on post-RM1 labels ---
# CRITICAL: RM2 must be calibrated on rm1_labels, NOT ws_labels
search_rm2 = list(range(50, 2001, 50))
all_mi2, all_var2, seg_counts2 = [], [], []

for t in search_rm2:
    labels = rm2(rm1_labels, filtered, cost_threshold=float(t))
    all_mi2.append(morans_i(labels, img))
    all_var2.append(intra_variance(labels, img))
    seg_counts2.append(int(labels.max()))

mi_arr2 = np.array(all_mi2).T
var_arr2 = np.array(all_var2).T
g2_rm2 = goodness2(mi_arr2, var_arr2)

best_rm2_idx = int(np.argmin(g2_rm2))
best_rm2 = search_rm2[best_rm2_idx]
print(f'Auto-selected RM2 threshold: {best_rm2} (G2={g2_rm2[best_rm2_idx]:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(search_rm2, g2_rm2, 'b-o', markersize=3)
axes[0].axvline(x=best_rm2, color='r', linewidth=2, label=f'Auto-selected={best_rm2}')
axes[0].set_xlabel('RM2 cost threshold')
axes[0].set_ylabel('G2 (lower = better)')
axes[0].set_title('G2 curve — RM2 threshold sweep (on post-RM1 labels)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(search_rm2, seg_counts2, 'g-s', markersize=3)
axes[1].axvline(x=best_rm2, color='r', linewidth=2, label=f'Auto-selected={best_rm2}')
axes[1].set_xlabel('RM2 cost threshold')
axes[1].set_ylabel('Number of segments')
axes[1].set_title('Segment count vs. RM2 threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Final result: RM3 with auto-selected thresholds ---
auto_labels = rm3(ws_labels, img, size_threshold=best_rm1, cost_threshold=best_rm2)
manual_labels = rm3(ws_labels, img, size_threshold=10, cost_threshold=200)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img.astype(np.uint8))
axes[0].set_title('Original (noisy)')
axes[0].axis('off')
axes[1].imshow(ws_labels, cmap='tab20')
axes[1].set_title(f'Watershed\n({ws_labels.max()} segs)')
axes[1].axis('off')
axes[2].imshow(manual_labels, cmap='tab20')
axes[2].set_title(f'RM3 manual\n(s={10}, c={200}) → {manual_labels.max()} segs')
axes[2].axis('off')
axes[3].imshow(auto_labels, cmap='tab20')
axes[3].set_title(f'RM3 auto\n(s={best_rm1}, c={best_rm2}) → {auto_labels.max()} segs')
axes[3].axis('off')
plt.suptitle('Auto-selected vs. manual thresholds', fontsize=13)
plt.tight_layout()
plt.show()